<a href="https://colab.research.google.com/github/Adan1816/bert-sentiment-analysis/blob/main/bert_sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install torch datasets transformers evaluate

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 3050 Laptop GPU


##LOADING THE DATASET

In [4]:
from datasets import load_dataset

dataset = load_dataset("imdb")
print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and

##LOADING THE TOKENIZER

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

sample = dataset["train"][0]["text"]
encoded = tokenizer(sample, padding="max_length", truncation=True, max_length=256, return_tensors="pt")

print(encoded.keys())
print(encoded["input_ids"].shape)
print(encoded["attention_mask"].shape)

a:\Work\bert-sentiment-analysis\venv\Lib\site-packages\huggingface_hub\file_download.py:157: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\adars\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
a:\Work\bert-sentiment-analysis\venv\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarni

dict_keys(['input_ids', 'attention_mask'])
torch.Size([1, 256])
torch.Size([1, 256])


In [6]:
print(tokenizer.decode(encoded["input_ids"][0][:30]))

[CLS] i rented i am curious - yellow from my video store because of all the controversy that surrounded it when it was first released in 1967. i also


##TOKENIZE THE FULL DATASET

In [7]:
def tokenize_function(examples):
  return tokenizer(
      examples["text"],
      padding="max_length",
      truncation=True,
      max_length=256,
  )

tokenized_dataset = dataset.map(tokenize_function, batched=True)
print(tokenized_dataset)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map: 100%|██████████| 50000/50000 [00:18<00:00, 2765.80 examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 50000
    })
})


In [8]:
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format("torch")

print(tokenized_dataset["train"].column_names)

['labels', 'input_ids', 'attention_mask']


##BUILD THE DATALOADER

In [9]:
from torch.utils.data import DataLoader

train_loader = DataLoader(tokenized_dataset["train"], shuffle=True, batch_size = 16)
val_loader = DataLoader(tokenized_dataset["test"], batch_size = 16)

In [10]:
batch = next(iter(train_loader))
print(batch.keys())
print(batch["input_ids"].shape)       # torch.Size([16, 256])
print(batch["attention_mask"].shape)  # torch.Size([16, 256])
print(batch["labels"].shape)          # torch.Size([16])
print(batch["labels"])                # mix of 0s and 1s

dict_keys(['labels', 'input_ids', 'attention_mask'])
torch.Size([16, 256])
torch.Size([16, 256])
torch.Size([16])
tensor([1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0])


In [12]:
import torch
from transformers import AutoModelForSequenceClassification, get_scheduler
from torch.optim import AdamW
from tqdm.auto import tqdm
import evaluate

# 1. Load model
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Training on: {device}")

# 2. Optimizer + scheduler
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

num_epochs = 3
num_training_steps = num_epochs * len(train_loader)

lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=int(0.1 * num_training_steps),
    num_training_steps=num_training_steps
)

# 3. Metric
metric = evaluate.load("accuracy")

# 4. Training loop
for epoch in range(num_epochs):
    # --- Train ---
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    # --- Evaluate ---
    model.eval()
    for batch in tqdm(val_loader, desc=f"Epoch {epoch+1} [Eval]"):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)

        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)
        metric.add_batch(predictions=predictions, references=batch["labels"])

    acc = metric.compute()
    print(f"\nEpoch {epoch+1} | Loss: {avg_loss:.4f} | Accuracy: {acc['accuracy']:.4f}\n")

# 5. Save model
model.save_pretrained("./bert-sentiment-analysis")
tokenizer.save_pretrained("./bert-sentiment-analysis")
print("Model saved.")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training on: cuda


Epoch 1 [Eval]: 100%|██████████| 1563/1563 [07:59<00:00,  3.26it/s]



Epoch 1 | Loss: 0.3122 | Accuracy: 0.9098



Epoch 2 [Eval]: 100%|██████████| 1563/1563 [08:08<00:00,  3.20it/s]



Epoch 2 | Loss: 0.1648 | Accuracy: 0.9148



Epoch 3 [Eval]: 100%|██████████| 1563/1563 [08:00<00:00,  3.25it/s]



Epoch 3 | Loss: 0.0805 | Accuracy: 0.9135

Model saved.


In [13]:
from sklearn.metrics import classification_report
import numpy as np

all_preds = []
all_labels = []

model.eval()
for batch in tqdm(val_loader, desc="Final Evaluation"):
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)
    preds = torch.argmax(outputs.logits, dim=-1)
    all_preds.extend(preds.cpu().numpy())
    all_labels.extend(batch["labels"].cpu().numpy())

print(classification_report(
    all_labels,
    all_preds,
    target_names=["Negative", "Positive"]
))

Final Evaluation: 100%|██████████| 1563/1563 [05:08<00:00,  5.07it/s]


              precision    recall  f1-score   support

    Negative       0.92      0.91      0.91     12500
    Positive       0.91      0.92      0.91     12500

    accuracy                           0.91     25000
   macro avg       0.91      0.91      0.91     25000
weighted avg       0.91      0.91      0.91     25000

